# Demo Setup

This notebook creates the schema and tables used in **Demo: EXPLAIN, Query Profile and Query Insights UI**.

In [0]:
import re

username = spark.sql("SELECT current_user()").first()[0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog_name = spark.sql("SELECT current_catalog()").first()[0]
schema_name = f"demo_query_perf_{clean_username}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}`")
spark.sql(f"USE CATALOG `{catalog_name}`")
spark.sql(f"USE SCHEMA `{schema_name}`")

In [0]:
# Create sales_fact using SQL for fast generation (50K rows)
spark.sql("DROP TABLE IF EXISTS sales_fact")
spark.sql("""
CREATE TABLE sales_fact AS
SELECT 
  CAST(id + 1 AS INT) AS sale_id,
  DATE_ADD('2023-01-01', CAST(RAND() * 729 AS INT)) AS sale_date,
  CASE CAST(FLOOR(RAND() * 4) AS INT) WHEN 0 THEN 'North' WHEN 1 THEN 'South' WHEN 2 THEN 'East' ELSE 'West' END AS region,
  CASE CAST(FLOOR(RAND() * 5) AS INT) WHEN 0 THEN 'Electronics' WHEN 1 THEN 'Clothing' WHEN 2 THEN 'Home' WHEN 3 THEN 'Food' ELSE 'Sports' END AS product_category,
  CASE CAST(FLOOR(RAND() * 3) AS INT) WHEN 0 THEN 'online' WHEN 1 THEN 'in_store' ELSE 'wholesale' END AS channel,
  ROUND(RAND() * 495 + 5, 2) AS amount,
  CAST(FLOOR(RAND() * 9) + 1 AS INT) AS quantity,
  CAST(FLOOR(RAND() * 5000) + 1 AS INT) AS customer_id
FROM RANGE(50000)
""")

count = spark.sql("SELECT COUNT(*) FROM sales_fact").first()[0]

In [0]:
# Create dim_products - small dimension table (triggers BroadcastHashJoin)
spark.sql("DROP TABLE IF EXISTS dim_products")
spark.sql("""
CREATE TABLE dim_products AS
SELECT * FROM VALUES
  (1, 'Electronics', 'Laptop', 450.00),
  (2, 'Electronics', 'Phone', 350.00),
  (3, 'Electronics', 'Tablet', 280.00),
  (4, 'Electronics', 'Headphones', 80.00),
  (5, 'Clothing', 'Jacket', 120.00),
  (6, 'Clothing', 'Shirt', 45.00),
  (7, 'Clothing', 'Shoes', 90.00),
  (8, 'Clothing', 'Hat', 25.00),
  (9, 'Home', 'Lamp', 65.00),
  (10, 'Home', 'Rug', 150.00),
  (11, 'Home', 'Blender', 85.00),
  (12, 'Home', 'Pillow', 30.00),
  (13, 'Food', 'Coffee', 15.00),
  (14, 'Food', 'Snacks', 8.00),
  (15, 'Food', 'Sauce', 6.00),
  (16, 'Food', 'Pasta', 4.00),
  (17, 'Sports', 'Bike', 400.00),
  (18, 'Sports', 'Ball', 25.00),
  (19, 'Sports', 'Weights', 60.00),
  (20, 'Sports', 'Mat', 35.00)
AS t(product_id, product_category, product_name, cost_price)
""")


In [0]:
print("SETUP COMPLETE")
print(f"")
print(f"Catalog:  {catalog_name}")
print(f"Schema:   {schema_name}")
print(f"")
print(f"Tables created:")
print(f"sales_fact    - 50,000 rows (2 years of daily sales)")
print(f"dim_products  - 20 rows (small dimension for JOIN demos)")